# 👋 PyCaret Anomaly Detection Tutorial

In [8]:
import duckdb
# check installed version
import pycaret
pycaret.__version__

'3.3.2'

In [4]:
def clear_disp():
    res = pull()
    display(
        res.style
        .set_properties(**{
            "color": "#111",
            "background-color": "#fff"
        })
    )

def esql(query):
    return duckdb.sql(query).df()

# 🚀 Quick start

PyCaret’s Anomaly Detection Module is an unsupervised machine learning module that is used for identifying rare items, events, or observations that raise suspicions by differing significantly from the majority of the data. 
Typically, the anomalous items will translate to some kind of problems such as bank fraud, a structural defect, medical problems, or errors. 

**Setup** ➡️ **Create Model** ➡️ **Assign Labels** ➡️ **Analyze Model** ➡️ **Prediction** ➡️ **Save Model**

In [1]:
# loading sample dataset from pycaret dataset module
from pycaret.datasets import get_data
data = get_data('anomaly')

,Col1,Col2,Col3,Col4,Col5,Col6,Col7,Col8,Col9,Col10
0,0.263995,0.764929,0.138424,0.935242,0.605867,0.518790,0.912225,0.608234,0.723782,0.733591
1,0.546092,0.653975,0.065575,0.227772,0.845269,0.837066,0.272379,0.331679,0.429297,0.367422
2,0.336714,0.538842,0.192801,0.553563,0.074515,0.332993,0.365792,0.861309,0.899017,0.088600
3,0.092108,0.995017,0.014465,0.176371,0.241530,0.514724,0.562208,0.158963,0.073715,0.208463
4,0.325261,0.805968,0.957033,0.331665,0.307923,0.355315,0.501899,0.558449,0.885169,0.182754


## Setup
This function initializes the training environment and creates the transformation pipeline. The setup function must be called before executing any other function. It takes one mandatory parameter only: data. All the other parameters are optional.

In [4]:
# import pycaret anomaly and init setup
from pycaret.anomaly import *
s = setup(data, session_id = 123,verbose=False)
clear_disp()

,Description,Value
0,Session id,123
1,Original data shape,"(1000, 10)"
2,Transformed data shape,"(1000, 10)"
3,Numeric features,10
4,Preprocess,True
5,Imputation type,simple
6,Numeric imputation,mean
7,Categorical imputation,mode
8,CPU Jobs,-1
9,Use GPU,False


## Create Model

This function trains an unsupervised anomaly detection model. All the available models can be accessed using the models function.

In [6]:
# train iforest model
iforest = create_model('iforest')
iforest

IForest(behaviour='new', bootstrap=False, contamination=0.05,
    max_features=1.0, max_samples='auto', n_estimators=100, n_jobs=-1,
    random_state=123, verbose=0)

In [7]:
# to check all the available models
models()

,Name,Reference
ID,,
abod,Angle-base Outlier Detection,pyod.models.abod.ABOD
cluster,Clustering-Based Local Outlier,pycaret.internal.patches.pyod.CBLOFForceToDouble
cof,Connectivity-Based Local Outlier,pyod.models.cof.COF
iforest,Isolation Forest,pyod.models.iforest.IForest
histogram,Histogram-based Outlier Detection,pyod.models.hbos.HBOS
knn,K-Nearest Neighbors Detector,pyod.models.knn.KNN
lof,Local Outlier Factor,pyod.models.lof.LOF
svm,One-class SVM detector,pyod.models.ocsvm.OCSVM
pca,Principal Component Analysis,pyod.models.pca.PCA


## Assign Model
This function assigns anomaly labels to the training data, given a trained model.

In [8]:
iforest_anomalies = assign_model(iforest)
iforest_anomalies

,Col1,Col2,Col3,Col4,Col5,Col6,Col7,Col8,Col9,Col10,Anomaly,Anomaly_Score
0,0.263995,0.764929,0.138424,0.935242,0.605867,0.518790,0.912225,0.608234,0.723782,0.733591,0,-0.035865
1,0.546092,0.653975,0.065575,0.227772,0.845269,0.837066,0.272379,0.331679,0.429297,0.367422,0,-0.084927
2,0.336714,0.538842,0.192801,0.553563,0.074515,0.332993,0.365792,0.861309,0.899017,0.088600,1,0.025356
3,0.092108,0.995017,0.014465,0.176371,0.241530,0.514724,0.562208,0.158963,0.073715,0.208463,1,0.042415
4,0.325261,0.805968,0.957033,0.331665,0.307923,0.355315,0.501899,0.558449,0.885169,0.182754,0,-0.023408
...,...,...,...,...,...,...,...,...,...,...,...,...
995,0.305055,0.656837,0.331665,0.822525,0.907127,0.882276,0.855732,0.584786,0.808640,0.242762,0,-0.083981
996,0.812627,0.864258,0.616604,0.167966,0.811223,0.938071,0.418462,0.472306,0.348347,0.671129,0,-0.075839
997,0.250967,0.138627,0.919703,0.461234,0.886555,0.869888,0.800908,0.530324,0.779433,0.234952,0,-0.052903
998,0.502436,0.936820,0.580062,0.540773,0.151995,0.059452,0.225220,0.242755,0.279385,0.538755,0,-0.075104


In [17]:
esql("select anomaly, min(anomaly_score),mean(anomaly_score),max(anomaly_score),count(*) from iforest_anomalies group by anomaly")

,Anomaly,min(anomaly_score),mean(anomaly_score),max(anomaly_score),count_star()
0,1,0.000349,0.023835,0.064037,50
1,0,-0.132984,-0.067232,-0.000018,950


## Analyze Model

You can use the `plot_model` function to analyzes the performance of a trained model on the test set. It may require re-training the model in certain cases.

In [18]:
# tsne plot anomalies
plot_model(iforest, plot = 'tsne')

## Prediction
The `predict_model` function returns `Anomaly` and `Anomaly_Score` label as a new column in the input dataframe. This step may or may not be needed depending on the use-case. Some times clustering models are trained for analysis purpose only and the interest of user is only in assigned labels on the training dataset, that can be done using `assign_model` function. `predict_model` is only useful when you want to obtain cluster labels on unseen data (i.e. data that was not used during training the model).

In [20]:
# predict on test set
iforest_pred = predict_model(iforest, data=data)
iforest_pred

,Col1,Col2,Col3,Col4,Col5,Col6,Col7,Col8,Col9,Col10,Anomaly,Anomaly_Score
0,0.263995,0.764929,0.138424,0.935242,0.605867,0.518790,0.912225,0.608234,0.723782,0.733591,0,-0.035865
1,0.546092,0.653975,0.065575,0.227772,0.845269,0.837066,0.272379,0.331679,0.429297,0.367422,0,-0.084927
2,0.336714,0.538842,0.192801,0.553563,0.074515,0.332993,0.365792,0.861309,0.899017,0.088600,1,0.025356
3,0.092108,0.995017,0.014465,0.176371,0.241530,0.514724,0.562208,0.158963,0.073715,0.208463,1,0.042415
4,0.325261,0.805968,0.957033,0.331665,0.307923,0.355315,0.501899,0.558449,0.885169,0.182754,0,-0.023408
...,...,...,...,...,...,...,...,...,...,...,...,...
995,0.305055,0.656837,0.331665,0.822525,0.907127,0.882276,0.855732,0.584786,0.808640,0.242762,0,-0.083981
996,0.812627,0.864258,0.616604,0.167966,0.811223,0.938071,0.418462,0.472306,0.348347,0.671129,0,-0.075839
997,0.250967,0.138627,0.919703,0.461234,0.886555,0.869888,0.800908,0.530324,0.779433,0.234952,0,-0.052903
998,0.502436,0.936820,0.580062,0.540773,0.151995,0.059452,0.225220,0.242755,0.279385,0.538755,0,-0.075104


In [21]:
esql("select anomaly, min(anomaly_score),mean(anomaly_score),max(anomaly_score),count(*) from iforest_pred group by anomaly")

,Anomaly,min(anomaly_score),mean(anomaly_score),max(anomaly_score),count_star()
0,0,-0.132984,-0.067232,-0.000018,950
1,1,0.000349,0.023835,0.064037,50


In [6]:
from pycaret.anomaly import *

exp_ano = setup(data=data, session_id=123,verbose=False)
iforest = create_model('iforest', fraction=0.01)

results = assign_model(iforest,verbose=False)
clear_disp()

,Description,Value
0,Session id,123
1,Original data shape,"(1000, 10)"
2,Transformed data shape,"(1000, 10)"
3,Numeric features,10
4,Preprocess,True
5,Imputation type,simple
6,Numeric imputation,mean
7,Categorical imputation,mode
8,CPU Jobs,-1
9,Use GPU,False


In [10]:
esql("select anomaly, min(anomaly_score),mean(anomaly_score),max(anomaly_score),count(*) from results group by anomaly")

,Anomaly,min(anomaly_score),mean(anomaly_score),max(anomaly_score),count_star()
0,0,-0.171985,-0.102787,-0.000010,990
1,1,0.000986,0.007970,0.025036,10


In [11]:
exp_ano = setup(data=data, session_id=123,verbose=False)
iforest = create_model('iforest', fraction=0.005)

results = assign_model(iforest,verbose=False)
clear_disp()

,Description,Value
0,Session id,123
1,Original data shape,"(1000, 10)"
2,Transformed data shape,"(1000, 10)"
3,Numeric features,10
4,Preprocess,True
5,Imputation type,simple
6,Numeric imputation,mean
7,Categorical imputation,mode
8,CPU Jobs,-1
9,Use GPU,False


In [12]:
esql("select anomaly, min(anomaly_score),mean(anomaly_score),max(anomaly_score),count(*) from results group by anomaly")

,Anomaly,min(anomaly_score),mean(anomaly_score),max(anomaly_score),count_star()
0,1,0.001791,0.010841,0.021613,5
1,0,-0.175408,-0.105686,-0.000009,995
